In [ ]:
═══════════════════════════════════════════════════════════════════
VNDHR Preprocessing Pipeline
Based on: "VNDHR: Variational Single Nighttime Image Dehazing for 
Enhancing Visibility in Intelligent Transportation Systems via 
Hybrid Regularization" (IEEE TITS 2025)

This code preprocesses nighttime hazy images using the VNDHR method
described in the paper. It processes both training and testing datasets.
═══════════════════════════════════════════════════════════════════
"""

import os
import glob
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy import sparse
from scipy.sparse.linalg import cg, LinearOperator

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════

# Input directories (raw hazy images)
TRAIN_INPUT = r"D:\Downloads\data\train\input"
TEST_INPUT = r"D:\Downloads\data\test\input"

# Output directories (VNDHR preprocessed)
TRAIN_VNDHR = r"D:\Downloads\data\train_vndhr"
TEST_VNDHR = r"D:\Downloads\data\test_vndhr"

# Create output directories
for directory in [TRAIN_VNDHR, TEST_VNDHR]:
    os.makedirs(directory, exist_ok=True)

# VNDHR Parameters (from paper Section IV-A)
VNDHR_PARAMS = {
    'lambda1': 0.002,    # ℓp norm weight for illumination
    'lambda2': 0.0001,   # Weighted ℓ2 norm for reflectance
    'lambda3': 0.001,    # ℓ1 TV norm for noise suppression
    'p': 0.65,           # Fractional-order norm parameter
    'max_iter': 10,      # Maximum iterations
    'tau1': 1e-6,        # Small constant for gradient magnitude
    'tau2': 1e-6,        # Small constant for gradient magnitude
    'epsilon': 0.001     # Convergence threshold
}

IMG_SIZE = 256  # Resize for computational efficiency

print("═"*80)
print("VNDHR PREPROCESSING PIPELINE")
print("Based on IEEE TITS 2025 Paper")
print("═"*80)
print(f"Parameters: λ1={VNDHR_PARAMS['lambda1']}, λ2={VNDHR_PARAMS['lambda2']}, "
      f"λ3={VNDHR_PARAMS['lambda3']}, p={VNDHR_PARAMS['p']}")
print("═"*80 + "\n")

# ═══════════════════════════════════════════════════════════════════
# GRADIENT OPERATORS (from paper Section III-B)
# ═══════════════════════════════════════════════════════════════════

def gradient_operators(H, W):
    """
    Construct discrete gradient operators using Toeplitz matrices
    Returns: Dx, Dy (sparse matrices for horizontal and vertical gradients)
    """
    N = H * W
    
    # Horizontal gradient (Dx)
    rowi, coli, di = [], [], []
    for r in range(H):
        for c in range(W):
            idx = r * W + c
            if c < W - 1:
                right = r * W + (c + 1)
                rowi.extend([idx, idx])
                coli.extend([idx, right])
                di.extend([-1, 1])
    Dx = sparse.coo_matrix((di, (rowi, coli)), shape=(N, N)).tocsr()
    
    # Vertical gradient (Dy)
    rowi, coli, di = [], [], []
    for r in range(H):
        for c in range(W):
            idx = r * W + c
            if r < H - 1:
                down = (r + 1) * W + c
                rowi.extend([idx, idx])
                coli.extend([idx, down])
                di.extend([-1, 1])
    Dy = sparse.coo_matrix((di, (rowi, coli)), shape=(N, N)).tocsr()
    
    return Dx, Dy

# ═══════════════════════════════════════════════════════════════════
# DIAGONAL PRECONDITIONER (for fast PCG solver)
# ═══════════════════════════════════════════════════════════════════

class DiagonalPreconditioner(LinearOperator):
    """Diagonal preconditioner for conjugate gradient solver"""
    def __init__(self, A):
        self.shape = A.shape
        self.dtype = A.dtype
        self.diag_inv = 1.0 / (A.diagonal() + 1e-8)
    
    def _matvec(self, x):
        return self.diag_inv * x

# ═══════════════════════════════════════════════════════════════════
# ENHANCED VNDHR VARIATIONAL MODEL (Algorithm 1 from paper)
# ═══════════════════════════════════════════════════════════════════

class EnhancedVNDHRVariational:
    """
    Implements the Hybrid Variational Model (HVM) from Section III-B
    Decomposes image into illumination and reflectance using:
    - ℓp norm for structure-aware illumination
    - Weighted ℓ2 norm for fine structures in reflectance  
    - ℓ1 TV norm for noise suppression
    """
    
    def __init__(self, img_np, params=None):
        self.S = img_np.astype(np.float32)
        self.H, self.W = self.S.shape[:2]
        
        # Load parameters
        if params is None:
            params = VNDHR_PARAMS
        self.p = params.get('p', 0.65)
        self.lambda1 = params.get('lambda1', 0.002)
        self.lambda2 = params.get('lambda2', 0.0001)
        self.lambda3 = params.get('lambda3', 0.001)
        self.max_iter = params.get('max_iter', 10)
        self.tau1 = params.get('tau1', 1e-6)
        self.tau2 = params.get('tau2', 1e-6)
        
        # Convert to HSV and extract V-channel (as per paper)
        hsv = cv2.cvtColor((self.S * 255).astype(np.uint8), 
                          cv2.COLOR_RGB2HSV).astype(np.float32) / 255.0
        self.I = hsv[:, :, 2].copy()  # Illumination (V-channel)
        self.R = np.ones_like(self.I)  # Reflectance initialization
        
        # Construct gradient operators
        self.Dx, self.Dy = gradient_operators(self.H, self.W)
    
    def compute_GI(self, I):
        """Compute weight matrix GI for ℓp norm (Eq. 8)"""
        gradIx = self.Dx.dot(I.flatten()).reshape(self.H, self.W)
        gradIy = self.Dy.dot(I.flatten()).reshape(self.H, self.W)
        magI = np.sqrt(gradIx**2 + gradIy**2)
        return np.power(np.maximum(magI, self.tau1), self.p - 2)
    
    def compute_WR(self, R):
        """Compute weight matrix WR for weighted ℓ2 norm (Eq. 6)"""
        gradRx = self.Dx.dot(R.flatten()).reshape(self.H, self.W)
        gradRy = self.Dy.dot(R.flatten()).reshape(self.H, self.W)
        magR = np.sqrt(gradRx**2 + gradRy**2)
        return 1.0 / (np.maximum(magR, self.tau2) + 1e-8)
    
    def compute_GR(self, R):
        """Compute weight matrix GR for ℓ1 norm (Eq. 9)"""
        gradRx = self.Dx.dot(R.flatten()).reshape(self.H, self.W)
        gradRy = self.Dy.dot(R.flatten()).reshape(self.H, self.W)
        magR = np.sqrt(gradRx**2 + gradRy**2)
        return np.maximum(magR, self.tau2) ** (-1)
    
    def solve_I_subproblem(self, R, GI):
        """Solve I sub-problem using PCG (Eq. 13)"""
        Rdiag = sparse.diags(R.flatten())
        Wgi = sparse.diags(GI.flatten())
        U = self.Dx.T.dot(Wgi.dot(self.Dx)) + self.Dy.T.dot(Wgi.dot(self.Dy))
        
        A_I = Rdiag.T.dot(Rdiag) + self.lambda1 * U
        b_I = Rdiag.T.dot(self.I.flatten())
        
        M = DiagonalPreconditioner(A_I)
        x0 = self.I.flatten()
        
        x_sol, info = cg(A_I, b_I, x0=x0, M=M, atol=1e-5, rtol=1e-5, maxiter=300)
        if info != 0:
            x_sol = x0
        
        return x_sol.reshape(self.H, self.W)
    
    def solve_R_subproblem(self, I, WR, GR):
        """Solve R sub-problem using PCG (Eq. 17)"""
        Idiag = sparse.diags(I.flatten())
        Wwr = sparse.diags(WR.flatten())
        Wgr = sparse.diags(GR.flatten())
        
        V = self.Dx.T.dot(Wwr.dot(self.Dx)) + self.Dy.T.dot(Wwr.dot(self.Dy))
        M_term = self.Dx.T.dot(Wgr.dot(self.Dx)) + self.Dy.T.dot(Wgr.dot(self.Dy))
        
        LHS_R = Idiag.T.dot(Idiag) + self.lambda2 * V + self.lambda3 * M_term
        b_R = Idiag.T.dot(I.flatten())
        
        M_prec = DiagonalPreconditioner(LHS_R)
        r0 = self.R.flatten()
        
        r_sol, info = cg(LHS_R, b_R, x0=r0, M=M_prec, atol=1e-5, rtol=1e-5, maxiter=300)
        if info != 0:
            r_sol = r0
        
        return np.clip(r_sol.reshape(self.H, self.W), 0, 2.5)
    
    def run(self):
        """Run iterative optimization (Algorithm 1)"""
        for iteration in range(self.max_iter):
            # Compute weight matrices
            GI = self.compute_GI(self.I)
            WR = self.compute_WR(self.R)
            GR = self.compute_GR(self.R)
            
            # Update I and R
            I_new = self.solve_I_subproblem(self.R, GI)
            R_new = self.solve_R_subproblem(I_new, WR, GR)
            
            # Check convergence
            if (np.linalg.norm(I_new - self.I) / np.linalg.norm(self.I) < 1e-3 and
                np.linalg.norm(R_new - self.R) / np.linalg.norm(self.R) < 1e-3):
                break
            
            self.I = I_new
            self.R = R_new
        
        return self.I, self.R

# ═══════════════════════════════════════════════════════════════════
# PREPROCESSING FUNCTION
# ═══════════════════════════════════════════════════════════════════

def process_vndhr_folder(input_dir, output_dir, img_size=IMG_SIZE):
    """
    Process all images in a folder using VNDHR
    
    Args:
        input_dir: Directory containing input hazy images
        output_dir: Directory to save VNDHR preprocessed results
        img_size: Resize dimension for processing
    """
    files = sorted(glob.glob(os.path.join(input_dir, "*.*")))
    
    if len(files) == 0:
        print(f"⚠️  No images found in {input_dir}")
        return 0
    
    print(f"Processing {len(files)} images from {os.path.basename(input_dir)}")
    success_count = 0
    
    for path in tqdm(files, desc=f"VNDHR → {os.path.basename(output_dir)}"):
        try:
            # Load and resize image
            img = Image.open(path).convert("RGB")
            img = img.resize((img_size, img_size), Image.LANCZOS)
            img_np = np.array(img) / 255.0
            
            # Apply VNDHR decomposition
            vndhr = EnhancedVNDHRVariational(img_np, params=VNDHR_PARAMS)
            I, R = vndhr.run()
            
            # Reconstruct VNDHR output: S' = I ◦ R
            out = np.clip(img_np * I[:, :, None] * R[:, :, None], 0, 1)
            
            # Save result
            base = os.path.basename(path)
            save_path = os.path.join(output_dir, base)
            Image.fromarray((out * 255).astype(np.uint8)).save(save_path, quality=95)
            
            success_count += 1
            
        except Exception as e:
            print(f"\n⚠️  Error processing {os.path.basename(path)}: {e}")
    
    print(f"✓ Successfully processed {success_count}/{len(files)} images\n")
    return success_count

# ═══════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("\n" + "═"*80)
    print("STEP 1: Processing Training Data")
    print("═"*80)
    train_count = process_vndhr_folder(TRAIN_INPUT, TRAIN_VNDHR, IMG_SIZE)
    
    print("═"*80)
    print("STEP 2: Processing Testing Data")
    print("═"*80)
    test_count = process_vndhr_folder(TEST_INPUT, TEST_VNDHR, IMG_SIZE)
    
    print("═"*80)
    print("VNDHR PREPROCESSING COMPLETE")
    print("═"*80)
    print(f"Training images processed: {train_count}")
    print(f"Testing images processed:  {test_count}")
    print(f"\nOutput directories:")
    print(f"  Train: {TRAIN_VNDHR}")
    print(f"  Test:  {TEST_VNDHR}")
    print("═"*80 + "\n")